
# Neural Style Transfer Demo

This notebook demonstrates the Neural Style Transfer (NST) technique using TensorFlow.
It uses a pre-trained VGG19 model to extract content and style features and then optimizes an image
to match the content of a content image and the style of a style image.

**Make sure you have  and  in the  directory relative to this notebook.**
You can use the sample images created in a previous step, or replace them with your own.


In [ ]:

import sys
import os

# Add src directory to Python path to import custom modules
# Assuming the notebook is in 'notebooks/' and src is in 'src/' at the same level as 'notebooks/'
module_path = os.path.abspath(os.path.join('../src'))
if module_path not in sys.path:
    sys.path.append(module_path)

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Import our utility and model functions
import utils
import nst_model

# Eager execution is enabled by default in TF 2.x
print(f"TensorFlow version: {tf.__version__}")
print(f"Eager execution: {tf.executing_eagerly()}")


## 1. Configuration and Setup

In [ ]:

# Define paths to content and style images
# These should be in the ../data/ directory relative to this notebook
CONTENT_IMAGE_PATH = os.path.join('..', 'data', 'content.jpg')
STYLE_IMAGE_PATH = os.path.join('..', 'data', 'style.jpg')

# Check if images exist
if not os.path.exists(CONTENT_IMAGE_PATH):
    print(f"ERROR: Content image not found at {CONTENT_IMAGE_PATH}")
if not os.path.exists(STYLE_IMAGE_PATH):
    print(f"ERROR: Style image not found at {STYLE_IMAGE_PATH}")

# Image dimensions
# The VGG19 model was trained on images of size 224x224.
# For style transfer, larger images can be used, but they require more memory and time.
# We use max_dim from utils.load_image to resize while maintaining aspect ratio.
MAX_DIM = 512 # You can reduce this if you run out of memory (e.g., 256)

# Weights for the loss components
CONTENT_WEIGHT = 1e3
STYLE_WEIGHT = 1e-2 # Can be 1e-2, 1e-1, 1.0 etc.
TOTAL_VARIATION_WEIGHT = 30 # Can be 10, 20, 30 etc.

# Training parameters
EPOCHS = 10 # Number of optimization iterations (epochs)
STEPS_PER_EPOCH = 100 # Number of steps per epoch
LEARNING_RATE = 0.02


## 2. Load and Preprocess Images

In [ ]:

# Load content and style images using our utility function
content_image_np = utils.load_image(CONTENT_IMAGE_PATH, max_dim=MAX_DIM)
style_image_np = utils.load_image(STYLE_IMAGE_PATH, max_dim=MAX_DIM)

if content_image_np is None or style_image_np is None:
    print("Please ensure content and style images are correctly loaded.")
else:
    # Preprocess images for VGG19
    # Our utils.load_image already normalizes to [0,1]
    # utils.preprocess_image_vgg handles VGG mean subtraction and BGR conversion
    preprocessed_content_image = utils.preprocess_image_vgg(content_image_np)
    preprocessed_style_image = utils.preprocess_image_vgg(style_image_np)

    print(f"Content image shape: {preprocessed_content_image.shape}")
    print(f"Style image shape: {preprocessed_style_image.shape}")

    # Function to display images (original [0,1] RGB or deprocessed VGG output)
    def imshow(image, title=None):
        if len(image.shape) > 3: # If batch dimension is present
            image = np.squeeze(image, axis=0)
        
        # If image is VGG preprocessed (BGR, mean subtracted), deprocess it first
        if image.min() < -100: # Heuristic: VGG preprocessed images have large negative values
             image_rgb = utils.deprocess_image_vgg(np.expand_dims(image, axis=0)) # deprocess needs batch
        elif image.max() <=1.0 and image.min() >=0: # Assumed to be [0,1] RGB
            image_rgb = (image * 255).astype(np.uint8)
        else: # Assumed to be already in displayable RGB [0,255] format
            image_rgb = image.astype(np.uint8)

        plt.imshow(image_rgb)
        if title:
            plt.title(title)
        plt.axis('off')

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    imshow(content_image_np, 'Content Image (Original [0,1])') # Display original [0,1] loaded image
    plt.subplot(1, 2, 2)
    imshow(style_image_np, 'Style Image (Original [0,1])')   # Display original [0,1] loaded image
    plt.show()


## 3. Define the Model and Extract Features

In [ ]:

# Create the StyleContentModel
# This model will return the intermediate layer outputs for content and style
extractor = nst_model.StyleContentModel(nst_model.STYLE_LAYERS, nst_model.CONTENT_LAYERS)

# Get target content and style representations
# These are computed once from the original content and style images
target_content_features = extractor(preprocessed_content_image)['content']
target_style_features = extractor(preprocessed_style_image)['style']


## 4. Initialize the Generated Image and Optimizer

In [ ]:

# Initialize the image to be optimized (the generated image)
# We can start from the content image or random noise. Starting from content image often converges faster.
# Ensure it's a tf.Variable as its pixels will be updated.
# The input to the extractor should be preprocessed.
generated_image_var = tf.Variable(preprocessed_content_image, dtype=tf.float32)

# Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=0.99, epsilon=1e-1)


## 5. Define the Optimization Step (Training Step)

In [ ]:

# Function to compute total loss
def compute_loss(model, loss_weights, init_image, target_content_features, target_style_features):
    model_outputs = model(init_image) # This is where the image is passed through VGG
    
    # Content Loss
    current_content_features = model_outputs['content']
    c_loss = nst_model.content_loss(current_content_features, target_content_features)
    
    # Style Loss
    current_style_features = model_outputs['style']
    s_loss = nst_model.style_loss(current_style_features, target_style_features) # Gram matrices computed inside
    
    # Total Variation Loss
    tv_loss = nst_model.total_variation_loss(init_image) # init_image is the generated image here
    
    # Weighted sum of losses
    total_loss = (loss_weights['content'] * c_loss + 
                  loss_weights['style'] * s_loss + 
                  loss_weights['total_variation'] * tv_loss)
    return total_loss, c_loss, s_loss, tv_loss

loss_weights_dict = {
    'content': CONTENT_WEIGHT, 
    'style': STYLE_WEIGHT, 
    'total_variation': TOTAL_VARIATION_WEIGHT
}

# Decorate with @tf.function for performance. 
# input_signature specifies the shape and type of the arguments to avoid re-tracing.
# The generated_image_var needs to match the shape of preprocessed_content_image.
@tf.function(input_signature=[
    tf.TensorSpec(shape=preprocessed_content_image.shape, dtype=tf.float32),
])
def train_step(image_variable):
    with tf.GradientTape() as tape:
        #  is the tf.Variable representing the generated image
        total_loss, content_l, style_l, tv_l = compute_loss(
            extractor, loss_weights_dict, image_variable, 
            target_content_features, target_style_features
        )
    
    gradients = tape.gradient(total_loss, image_variable)
    optimizer.apply_gradients([(gradients, image_variable)])
    # Clip image values to maintain valid pixel range (after VGG mean subtraction and BGR)
    # This clipping should consider the typical range of VGG preprocessed images.
    # For example, VGG inputs are often roughly in [-120, 120] after mean subtraction.
    # Clipping to [0, 255] then re-applying preprocessing, or clipping to preprocessed range.
    # For now, we clip based on the style image's preprocessed range as a heuristic.
    # A more robust way is to ensure deprocess_image handles clipping to valid display ranges.
    # image_variable.assign(tf.clip_by_value(image_variable, clip_value_min=-120, clip_value_max=120))
    
    return total_loss, content_l, style_l, tv_l


## 6. Run the Optimization Loop

In [ ]:

import time
start_time = time.time()

total_loss_history = []

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    epoch_start_time = time.time()
    for step in range(STEPS_PER_EPOCH):
        current_total_loss, c_loss, s_loss, tv_loss = train_step(generated_image_var)
        
        if (step + 1) % 20 == 0: # Print loss every 20 steps
            print(f"  Step {step + 1}/{STEPS_PER_EPOCH} - "
                  f"Total Loss: {current_total_loss:.2f}, "
                  f"Content Loss: {c_loss * CONTENT_WEIGHT:.2f}, "
                  f"Style Loss: {s_loss * STYLE_WEIGHT:.2f}, "
                  f"TV Loss: {tv_loss * TOTAL_VARIATION_WEIGHT:.2f}")
    
    total_loss_history.append(current_total_loss.numpy())
    print(f"Epoch {epoch + 1} completed in {time.time() - epoch_start_time:.2f}s. Last Total Loss: {current_total_loss:.2f}")
    
    # Display intermediate result (optional, can slow down training)
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS -1 : # Display every 5 epochs or at the end
        plt.figure(figsize=(5,5))
        # Deprocess the generated image for display
        # The generated_image_var is preprocessed (BGR, mean-subtracted)
        img_to_show = utils.deprocess_image_vgg(generated_image_var.numpy())
        imshow(img_to_show, title=f"Generated Image (Epoch {epoch + 1})")
        plt.show()

end_time = time.time()
print(f"Total optimization time: {end_time - start_time:.2f} seconds")

# Plot loss history
plt.figure(figsize=(8,5))
plt.plot(total_loss_history, label='Total Loss')
plt.title('Total Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


## 7. Display Final Results

In [ ]:

# Deprocess the final generated image for display
final_image_deprocessed = utils.deprocess_image_vgg(generated_image_var.numpy())

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
imshow(content_image_np, 'Content Image') # Original [0,1]
plt.subplot(1, 3, 2)
imshow(style_image_np, 'Style Image')   # Original [0,1]
plt.subplot(1, 3, 3)
imshow(final_image_deprocessed, 'Stylized Image')
plt.show()

# Save the final image (optional)
# cv2.imwrite('../data/stylized_output.jpg', cv2.cvtColor(final_image_deprocessed, cv2.COLOR_RGB2BGR))
# print("Saved final stylized image to ../data/stylized_output.jpg")



## Notes and Further Steps:

- **Hyperparameter Tuning:** The weights for content, style, and total variation loss, as well as the learning rate and number of iterations, significantly impact the result. Experiment with these values.
- **Different Layers:** Try using different layers for content and style extraction in .
- **Performance:** For faster results, consider reducing , , or .  helps, but NST is computationally intensive.
- **Advanced Techniques:** Explore Fast Neural Style Transfer models for real-time applications after an initial training phase.
